In [11]:
from pathlib import Path
import duckdb
import numpy as np

print("1. Fetching unique customer list from 'paySim dataset.csv'...")

# Fetch all unique origin customers
unique_cust_df = duckdb.query("""
    SELECT DISTINCT nameOrig AS customer_id 
    FROM read_csv_auto('paySim dataset.csv')
""").df()

n_cust = len(unique_cust_df)
np.random.seed(42)

# Generate synthetic profile columns
unique_cust_df['account_type'] = np.random.choice(['SAVINGS', 'CHECKING', 'BUSINESS'], size=n_cust)
unique_cust_df['customer_segment'] = np.random.choice(['RETAIL', 'VIP', 'CORPORATE'], size=n_cust)
unique_cust_df['kyc_status'] = np.random.choice(['VERIFIED', 'PENDING'], size=n_cust, p=[0.85, 0.15])
unique_cust_df['risk_rating_base'] = np.random.choice(['LOW', 'MEDIUM', 'HIGH'], size=n_cust, p=[0.60, 0.30, 0.10])

# Save as CSV to Desktop
output_path = Path.home() / "Desktop" / "customer_profiles.csv"
unique_cust_df.to_csv(output_path, index=False)

print(f"-> Success: 'customer_profiles.csv' created for {n_cust:,} customers!")

1. Fetching unique customer list from 'paySim dataset.csv'...
-> Success: 'customer_profiles.csv' created for 6,353,307 customers!


In [12]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

# STEP 2: Risk Analysis with DuckDB (SQL JOIN) and Pandas

sql_mule_query = """
SELECT 
    p.nameOrig AS customer_id,
    cp.account_type,
    cp.customer_segment,
    cp.kyc_status,
    cp.risk_rating_base,
    SUM(CASE WHEN p.type IN ('CASH_IN', 'TRANSFER') THEN p.amount ELSE 0 END) AS total_inflow,
    SUM(CASE WHEN p.type = 'PAYMENT' THEN p.amount ELSE 0 END) AS total_outflow,
    COUNT(p.amount) AS total_txn_count
FROM read_csv_auto('paySim dataset.csv') p
LEFT JOIN read_csv_auto('customer_profiles.csv') cp
       ON p.nameOrig = cp.customer_id
GROUP BY p.nameOrig, cp.account_type, cp.customer_segment, cp.kyc_status, cp.risk_rating_base
HAVING total_inflow >= 50000
ORDER BY total_inflow DESC
"""

print("[1/2] Executing SQL JOIN query via DuckDB...")
mule_df = duckdb.query(sql_mule_query).df()

print("[2/2] Calculating risk metrics using Pandas...")

# Calculate Drain Percentage (ratio of outflow to inflow)
mule_df['drain_percentage'] = np.where(
    mule_df['total_inflow'] > 0,
    (mule_df['total_outflow'] / mule_df['total_inflow']) * 100,
    0
)

# Assign Final Risk Level
def calculate_risk(row):
    if row['total_inflow'] >= 200000 or row['kyc_status'] == 'PENDING':
        return 'CRITICAL'
    elif row['total_inflow'] >= 100000 or row['risk_rating_base'] == 'HIGH':
        return 'HIGH'
    else:
        return 'MEDIUM'

mule_df['final_risk_level'] = mule_df.apply(calculate_risk, axis=1)

# Save results to Desktop as CSV
output_path = Path.home() / "Desktop" / "AML_Alerts_Money_Mule.csv"
mule_df.to_csv(output_path, index=False)
print("-> Completed! 'AML_Alerts_Money_Mule.csv' has been generated.")

# Display the first 5 rows
mule_df.head()

[1/2] Executing SQL JOIN query via DuckDB...
[2/2] Calculating risk metrics using Pandas...
-> Completed! 'AML_Alerts_Money_Mule.csv' has been generated.


,customer_id,account_type,customer_segment,kyc_status,risk_rating_base,total_inflow,total_outflow,total_txn_count,drain_percentage,final_risk_level
0,C1715283297,CHECKING,CORPORATE,VERIFIED,MEDIUM,92445516.64,0.0,1,0.0,CRITICAL
1,C2127282686,BUSINESS,VIP,VERIFIED,LOW,73823490.36,0.0,1,0.0,CRITICAL
2,C2044643633,CHECKING,CORPORATE,VERIFIED,HIGH,71172480.42,0.0,1,0.0,CRITICAL
3,C1425667947,CHECKING,RETAIL,VERIFIED,LOW,69886731.30,0.0,1,0.0,CRITICAL
4,C1584456031,CHECKING,VIP,VERIFIED,LOW,69337316.27,0.0,1,0.0,CRITICAL
